In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MBA-Transformation-Gold") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.pyspark.python", "/usr/bin/python3") \
    .config("spark.pyspark.driver.python", "/usr/bin/python3") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setLocalProperty("spark.scheduler.pool", "pool2")
print("SparkSession aktif:", spark.version)

SparkSession aktif: 3.5.0


In [2]:
# Membaca data dari Silver layer
SILVER_PATH = "s3a://bigdata/silver/retail_cleaned/"

df_clean = spark.read.parquet(SILVER_PATH)
print(f"Baris dibaca dari Silver: {df_clean.count()}")
df_clean.show(5)

Baris dibaca dari Silver: 1000000
+--------------+--------------------+----------------+------+
|Transaction_ID|             Product|Discount_Applied|Season|
+--------------+--------------------+----------------+------+
|    1000000000|['Ketchup', 'Shav...|               1|Winter|
|    1000000001|['Ice Cream', 'Mi...|               1|  Fall|
|    1000000004|       ['Dish Soap']|               0|Winter|
|    1000000006|['Honey', 'BBQ Sa...|               0|Summer|
|    1000000007|['Syrup', 'Trash ...|               0|Winter|
+--------------+--------------------+----------------+------+
only showing top 5 rows



In [3]:
from pyspark.sql import functions as F

# Transformasi ke format array
df_transformed = df_clean \
    .withColumn("Product", F.expr("regexp_replace(Product, '\\\\[', '')")) \
    .withColumn("Product", F.expr("regexp_replace(Product, '\\\\]', '')")) \
    .withColumn("Product", F.expr("regexp_replace(Product, \"\\\\'\" , '')")) \
    .withColumn("Product", F.expr("split(trim(Product), ', ')"))

# Basket Filter: Hapus transaksi item <= 1
df_basket = df_transformed.filter(F.expr("cardinality(Product) > 1"))

print("Data Transformation (Array format & Basket Filter) Selesai")
df_basket.show(5, truncate=False)
df_basket.printSchema()

Data Transformation (Array format & Basket Filter) Selesai
+--------------+---------------------------------------------------+----------------+------+
|Transaction_ID|Product                                            |Discount_Applied|Season|
+--------------+---------------------------------------------------+----------------+------+
|1000000000    |[Ketchup, Shaving Cream, Light Bulbs]              |1               |Winter|
|1000000001    |[Ice Cream, Milk, Olive Oil, Bread, Potatoes]      |1               |Fall  |
|1000000006    |[Honey, BBQ Sauce, Soda, Olive Oil, Garden Hose]   |0               |Summer|
|1000000007    |[Syrup, Trash Cans, Pancake Mix, Water, Mayonnaise]|0               |Winter|
|1000000011    |[Tea, Paper Towels, Spinach]                       |0               |Fall  |
+--------------+---------------------------------------------------+----------------+------+
only showing top 5 rows

root
 |-- Transaction_ID: string (nullable = true)
 |-- Product: array (nullabl

In [4]:
from pyspark.sql import functions as F

print("=== Quality Check ===")
print(f"Total transaksi: {df_basket.count()}")
df_basket.printSchema()

# Cek null kolom kunci
nulls = df_basket.filter(
    F.col("Transaction_ID").isNull() |
    F.col("Product").isNull() |
    F.col("Season").isNull() |
    F.col("Discount_Applied").isNull()
).count()
print(f"Null kritis: {nulls} baris")

# Distribusi Discount_Applied
print("=== Distribusi Discount_Applied ===")
df_basket.groupBy("Discount_Applied").count().show()

# Distribusi Season
print("=== Distribusi Season ===")
df_basket.groupBy("Season").count().orderBy("Season").show()

# Statistik jumlah item per basket
df_basket.withColumn("item_count", F.expr("cardinality(Product)")) \
    .select(
        F.min("item_count").alias("min_items"),
        F.max("item_count").alias("max_items"),
        F.avg("item_count").alias("avg_items")
    ).show()

=== Quality Check ===
Total transaksi: 800270
root
 |-- Transaction_ID: string (nullable = true)
 |-- Product: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- Discount_Applied: integer (nullable = true)
 |-- Season: string (nullable = true)

Null kritis: 0 baris
=== Distribusi Discount_Applied ===
+----------------+------+
|Discount_Applied| count|
+----------------+------+
|               0|399722|
|               1|400548|
+----------------+------+

=== Distribusi Season ===
+------+------+
|Season| count|
+------+------+
|  Fall|200244|
|Spring|200416|
|Summer|199419|
|Winter|200191|
+------+------+

+---------+---------+-----------------+
|min_items|max_items|        avg_items|
+---------+---------+-----------------+
|        2|        5|3.499585140015245|
+---------+---------+-----------------+



In [5]:
# Menyimpan data ke minio dengan partisi Season
GOLD_PATH = "s3a://bigdata/gold/basket_transactions/"

df_basket.write \
    .mode("overwrite") \
    .partitionBy("Season") \
    .parquet(GOLD_PATH)

print(f"✅ Gold layer tersimpan di: {GOLD_PATH}")
print("Partisi berdasarkan Season: Spring, Summer, Fall, Winter")

✅ Gold layer tersimpan di: s3a://bigdata/gold/basket_transactions/
Partisi berdasarkan Season: Spring, Summer, Fall, Winter
